# **LIBRARIES**

In [ ]:
!pip install groq


In [ ]:
%%time
import json
from google.colab import userdata
from groq import Groq
import spacy
from pydantic import BaseModel, Field

CPU times: user 21 µs, sys: 0 ns, total: 21 µs
Wall time: 24.3 µs


Environment

In [ ]:
api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)
nlp = spacy.load("en_core_web_sm")
print("Environment successfully initialized! Ready to build agents.")


Environment successfully initialized! Ready to build agents.


# **MATCHING AGENT**

Define the AI Output Rules

In [ ]:
class MatchDecision(BaseModel):
    match_found: bool = Field(description="True if a valid NGO match is found, otherwise False.")
    best_ngo_id: str = Field(description="The ngo_id of the most suitable NGO. Empty string if no match.")
    confidence_score: float = Field(description="A confidence score between 0.0 and 1.0.")
    reasoning: str = Field(description="Step-by-step logic explaining why this NGO was chosen over others based on capacity and distance.")
    dispatch_urgency: str = Field(description="'High', 'Medium', or 'Low', depending on the time_to_spoil_hours.")

Build the Matching Agent Logic


In [ ]:
def run_matching_agent_groq(surplus_data: dict, active_ngos: list) -> str:
    """Passes surplus data and available NGOs to Groq to find the optimal match."""
    if not client:
        return '{"error": "Client not initialized."}'

    # Convert our Pydantic rules into a schema string for the AI to read
    schema_instructions = json.dumps(MatchDecision.model_json_schema(), indent=2)

    system_prompt = f"""
    You are the Matching Agent for a food rescue logistics platform.
    You must output ONLY valid JSON.
    The JSON must strictly follow this exact schema structure:
    {schema_instructions}
    """

    user_prompt = f"""
    Analyze the available surplus food and the active NGOs.
    Find the best match based on capacity, accepted food types, and location.

    Surplus Food: {surplus_data}
    Available NGOs: {active_ngos}
    """

    # Call Groq's API using the blazing fast Llama 3.1 8B model
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"}, # Forces JSON output
        temperature=0.1,
    )

    return response.choices[0].message.content

Execute and Save JSON Output

In [ ]:
%%time

# --- Simulation Data ---
surplus = {
    "food_type": "Cooked Rice & Dal",
    "quantity_servings": 50,
    "time_to_spoil_hours": 3,
    "location_lat": 28.7041,
    "location_lon": 77.1025
}

ngos = [
    {"ngo_id": "NGO_001", "name": "Hope Foundation", "capacity_servings": 20, "accepts_cooked": True},
    {"ngo_id": "NGO_002", "name": "Delhi Hunger Relief", "capacity_servings": 100, "accepts_cooked": True}
]

# Execute the agent
if client:
    print("Agent is analyzing network data...\n")
    json_output = run_matching_agent_groq(surplus, ngos)

    # Display the formatted JSON result
    print(json.dumps(json.loads(json_output), indent=2))

Agent is analyzing network data...

{
  "match_found": true,
  "best_ngo_id": "NGO_002",
  "confidence_score": 0.9,
  "reasoning": "NGO_002 (Delhi Hunger Relief) has a higher capacity (100 servings) and accepts cooked food, making it the best match. Although NGO_001 (Hope Foundation) also accepts cooked food, its capacity is lower (20 servings).",
  "dispatch_urgency": "High"
}
CPU times: user 7.22 ms, sys: 1.94 ms, total: 9.16 ms
Wall time: 282 ms


# **VOLUNTEER DISPATCH**

Parse the Match and Prepare the Handoff

In [ ]:
import json

# 1. Parse the JSON string from our previous cell back into a Python dictionary
match_data = json.loads(json_output)

# 2. Check if a match was actually found before trying to dispatch
if not match_data.get("match_found"):
    print("No valid NGO match found. Halting dispatch process.")
else:
    best_ngo_id = match_data.get("best_ngo_id")
    urgency = match_data.get("dispatch_urgency")
    print(f"Handoff Successful: Preparing to dispatch for {best_ngo_id} with {urgency} urgency.")

Handoff Successful: Preparing to dispatch for NGO_002 with High urgency.


Define the AI Output Rules

In [ ]:
# Define the Dispatch Agent's strict JSON output
class DispatchDecision(BaseModel):
    assigned_volunteer_id: str = Field(description="The vol_id of the best volunteer. Empty if none suitable.")
    vehicle_type: str = Field(description="The type of vehicle assigned.")
    estimated_pickup_mins: int = Field(description="Estimated minutes for the volunteer to reach the surplus location.")
    reasoning: str = Field(description="Explanation of why this volunteer was chosen based on vehicle capacity and distance.")

Build the Volunteer Dispatch Logic

In [ ]:
def run_dispatch_agent_groq(surplus_data: dict, target_ngo_id: str, urgency: str, active_vols: list) -> str:
    """Selects the best volunteer for the pickup and delivery task using Groq."""
    if not client:
        return '{"error": "Client not initialized."}'

    # Convert our Pydantic rules into a schema string for the AI to read
    schema_instructions = json.dumps(DispatchDecision.model_json_schema(), indent=2)

    system_prompt = f"""
    You are the Dispatch Agent for a food rescue logistics platform.
    A match has been made. The food needs to be picked up from the Surplus Location and delivered to NGO {target_ngo_id}.
    Urgency level: {urgency}.

    Task: Choose the best active volunteer to pick up the food.
    Constraints:
    1. A 'bike' can carry a maximum of 30 servings.
    2. A 'car' can carry up to 150 servings.
    3. Prioritize volunteers who are closest to the Surplus Location.

    You must output ONLY valid JSON.
    The JSON must strictly follow this exact schema structure:
    {schema_instructions}
    """

    user_prompt = f"""
    Surplus Data (Pickup Location): {surplus_data}
    Available Volunteers: {active_vols}
    """

    # We continue using the fast llama-3.1-8b-instant model
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0.1,
    )
    return response.choices[0].message.content

Execute and Route the Driver

In [ ]:
%%time

# We use the surplus data from the previous cell (50 servings)
active_volunteers = [
    {
        "vol_id": "VOL_089",
        "name": "Aman",
        "vehicle": "bike", # Capacity 30 - Too small for 50 servings
        "current_lat": 28.7042,
        "current_lon": 77.1026, # Very close to surplus
        "status": "idle"
    },
    {
        "vol_id": "VOL_102",
        "name": "Priya",
        "vehicle": "car", # Capacity 150 - Can handle 50 servings
        "current_lat": 28.7060,
        "current_lon": 77.1040, # Slightly further away
        "status": "idle"
    }
]

# Only run if the previous cell found a match
if 'match_data' in locals() and match_data.get("match_found"):
    print("Dispatch Agent is calculating optimal route...\n")
    dispatch_output = run_dispatch_agent_groq(
        surplus_data=surplus,
        target_ngo_id=best_ngo_id,
        urgency=urgency,
        active_vols=active_volunteers
    )

    print(json.dumps(json.loads(dispatch_output), indent=2))
else:
    print("Cannot run dispatch: No match data available.")

Dispatch Agent is calculating optimal route...

{
  "assigned_volunteer_id": "VOL_102",
  "vehicle_type": "car",
  "estimated_pickup_mins": 5,
  "reasoning": "Volunteer VOL_102 is chosen because they are closest to the surplus location and have a car, which can carry more than the required 50 servings."
}
CPU times: user 4.86 ms, sys: 0 ns, total: 4.86 ms
Wall time: 346 ms


# **WHATSAPP INGESTION PIPELINE**

Load the NLP Engine

In [ ]:
import re
import json

# Load the local engine for grammatical analysis
nlp = spacy.load("en_core_web_sm")
print("✅ spaCy NLP Engine Loaded Successfully")

✅ spaCy NLP Engine Loaded Successfully


Define the Core Parsing Logic

In [ ]:
import re
import json

def parse_whatsapp_surplus(message_text: str, current_gps: dict = None) -> dict:
    """
    Extracts structured food type and quantity using Contextual Anchoring Regex
    instead of blind noun chunking.
    """
    text_lower = message_text.lower()

    quantity = 30 # Default operational fallback
    extracted_food = "Unspecified Food"

    # Smarter Regex: Captures (1) Quantity, (2) Unit, and (3) The Food Name
    # It stops capturing when it hits words like "remaining", "left", "from", etc.
    extraction_pattern = r'(\d+)\s*(plates|servings|serves|kg|boxes|portions|meals|containers|packets|trays)\s*(?:of\s+)?(.*?)(?=\s+(?:remaining|left|ready|from|extra|now|need|urgent|please)|\.|,|;|$)'

    match = re.search(extraction_pattern, text_lower)

    if match:
        quantity = int(match.group(1))
        # Group 3 safely contains the food text between the unit and the stop-words
        raw_food = match.group(3).strip()

        if raw_food:
            extracted_food = raw_food.title()

    # Construct structured payload expected by the Matching Agent
    structured_payload = {
        "food_type": extracted_food,
        "quantity_servings": quantity,
        "time_to_spoil_hours": 4,
        "location_lat": current_gps["lat"] if current_gps else 28.7041,
        "location_lon": current_gps["lon"] if current_gps else 77.1025
    }

    return structured_payload

Test with Unstructured WhatsApp Inputs

In [ ]:
# Simulated live incoming message streams along with metadata location pings
test_messages = [
    {
        "text": "Hey, we have around 40 plates of leftover chicken biryani ready.",
        "gps": {"lat": 28.7041, "lon": 77.1025}
    },
    {
        "text": "urgent: 15 boxes of paneer tikka and dal makhani remaining from the wedding party.",
        "gps": {"lat": 28.7120, "lon": 77.1200}
    },
    {
        "text": "Got 60 portions of fresh mixed veg fried rice extra tonight.",
        "gps": {"lat": 28.6985, "lon": 77.0912}
    }
]

print("Processing incoming real-time alerts...\n")

for i, incoming in enumerate(test_messages, start=1):
    parsed_result = parse_whatsapp_surplus(incoming["text"], incoming["gps"])

    print(f"--- Raw Message #{i} ---")
    print(f'"{incoming["text"]}"')
    print(f"--- Parsed Payload for Matching Agent ---")
    print(json.dumps(parsed_result, indent=2))
    print("-" * 40 + "\n")

Processing incoming real-time alerts...

--- Raw Message #1 ---
"Hey, we have around 40 plates of leftover chicken biryani ready."
--- Parsed Payload for Matching Agent ---
{
  "food_type": "Leftover Chicken Biryani",
  "quantity_servings": 40,
  "time_to_spoil_hours": 4,
  "location_lat": 28.7041,
  "location_lon": 77.1025
}
----------------------------------------

--- Raw Message #2 ---
"urgent: 15 boxes of paneer tikka and dal makhani remaining from the wedding party."
--- Parsed Payload for Matching Agent ---
{
  "food_type": "Paneer Tikka And Dal Makhani",
  "quantity_servings": 15,
  "time_to_spoil_hours": 4,
  "location_lat": 28.712,
  "location_lon": 77.12
}
----------------------------------------

--- Raw Message #3 ---
"Got 60 portions of fresh mixed veg fried rice extra tonight."
--- Parsed Payload for Matching Agent ---
{
  "food_type": "Fresh Mixed Veg Fried Rice",
  "quantity_servings": 60,
  "time_to_spoil_hours": 4,
  "location_lat": 28.6985,
  "location_lon": 77.0912

# **END-TO-END SIMULATION**

The Master Pipeline Orchestrator

In [ ]:
def run_food_rescue_pipeline(whatsapp_text: str, current_gps: dict, available_ngos: list, active_vols: list):
    print("="*70)
    print("🚨 NEW FOOD SURPLUS ALERT DETECTED")
    print("="*70)
    print(f"Incoming WhatsApp: '{whatsapp_text}'\n")

    # --- Step 1: Rule-Based NLP Extraction ---
    print(">> [Step 1] Initializing Local NLP Extractor...")
    surplus_data = parse_whatsapp_surplus(whatsapp_text, current_gps)
    print(f"   Extracted Payload: {json.dumps(surplus_data, indent=2)}\n")

    # --- Step 2: Groq Matching Agent ---
    print(">> [Step 2] Pinging Groq Matching Agent (llama-3.1-8b-instant)...")
    match_raw = run_matching_agent_groq(surplus_data, available_ngos)

    try:
        match_res = json.loads(match_raw)
    except json.JSONDecodeError:
        print("   ❌ Error: Matching Agent did not return valid JSON.")
        return

    print(f"   Match Found: {match_res.get('match_found')}")
    print(f"   Target NGO: {match_res.get('best_ngo_id')}")
    print(f"   Reasoning: {match_res.get('reasoning')}\n")

    if not match_res.get("match_found"):
        print("❌ Pipeline Terminated: No matching NGO has the capacity or accepts this food type.")
        print("="*70 + "\n")
        return

    # --- Step 3: Groq Volunteer Dispatch Agent ---
    best_ngo_id = match_res.get("best_ngo_id")
    urgency = match_res.get("dispatch_urgency")

    print(f">> [Step 3] Pinging Groq Dispatch Agent (Urgency: {urgency})...")
    dispatch_raw = run_dispatch_agent_groq(surplus_data, best_ngo_id, urgency, active_vols)

    try:
        dispatch_res = json.loads(dispatch_raw)
    except json.JSONDecodeError:
        print("   ❌ Error: Dispatch Agent did not return valid JSON.")
        return

    volunteer_id = dispatch_res.get('assigned_volunteer_id')

    # --- Final Output ---
    print("="*70)
    if volunteer_id:
        print(f"✅ SUCCESS: ROUTE OPTIMIZED & DISPATCHED")
        print(f"   Driver Assigned : {volunteer_id} ({dispatch_res.get('vehicle_type')})")
        print(f"   Pickup Target   : {surplus_data['food_type']} ({surplus_data['quantity_servings']} servings)")
        print(f"   Dropoff Target  : {best_ngo_id}")
        print(f"   ETA to Pickup   : ~{dispatch_res.get('estimated_pickup_mins')} mins")
        print(f"   Routing Logic   : {dispatch_res.get('reasoning')}")
    else:
        print(f"⚠️ ALERT: Match found, but no suitable volunteers are active to handle the capacity.")
    print("="*70 + "\n")

Live Execution & Simulation

In [ ]:
# --- Active Network Data ---
network_ngos = [
    {"ngo_id": "NGO_001", "name": "Hope Foundation", "capacity_servings": 20, "accepts_cooked": True, "lat": 28.7050, "lon": 77.1000},
    {"ngo_id": "NGO_002", "name": "Delhi Hunger Relief", "capacity_servings": 200, "accepts_cooked": True, "lat": 28.7100, "lon": 77.1150}
]

fleet_volunteers = [
    {"vol_id": "VOL_089", "name": "Aman", "vehicle": "bike", "current_lat": 28.7042, "current_lon": 77.1026, "status": "idle"},
    {"vol_id": "VOL_102", "name": "Priya", "vehicle": "car", "current_lat": 28.7060, "current_lon": 77.1040, "status": "idle"}
]

# --- Test Scenario 1: A massive wedding surplus ---
wedding_manager_text = "urgent: 150 boxes of paneer tikka and dal makhani remaining from the wedding party. need someone to pick this up now."
wedding_gps = {"lat": 28.7120, "lon": 77.1200}

# Execute Pipeline
run_food_rescue_pipeline(
    whatsapp_text=wedding_manager_text,
    current_gps=wedding_gps,
    available_ngos=network_ngos,
    active_vols=fleet_volunteers
)

# --- Test Scenario 2: A small cafe surplus ---
cafe_manager_text = "Hey, we have 15 portions of extra sandwiches left."
cafe_gps = {"lat": 28.6985, "lon": 77.0912}

# Execute Pipeline
run_food_rescue_pipeline(
    whatsapp_text=cafe_manager_text,
    current_gps=cafe_gps,
    available_ngos=network_ngos,
    active_vols=fleet_volunteers
)

🚨 NEW FOOD SURPLUS ALERT DETECTED
Incoming WhatsApp: 'urgent: 150 boxes of paneer tikka and dal makhani remaining from the wedding party. need someone to pick this up now.'

>> [Step 1] Initializing Local NLP Extractor...
   Extracted Payload: {
  "food_type": "Paneer Tikka And Dal Makhani",
  "quantity_servings": 150,
  "time_to_spoil_hours": 4,
  "location_lat": 28.712,
  "location_lon": 77.12
}

>> [Step 2] Pinging Groq Matching Agent (llama-3.1-8b-instant)...
   Match Found: True
   Target NGO: NGO_002
   Reasoning: NGO_002 (Delhi Hunger Relief) has the highest capacity to serve 200 people and is located closest to the surplus food. It also accepts cooked food.

>> [Step 3] Pinging Groq Dispatch Agent (Urgency: High)...
✅ SUCCESS: ROUTE OPTIMIZED & DISPATCHED
   Driver Assigned : VOL_102 (car)
   Pickup Target   : Paneer Tikka And Dal Makhani (150 servings)
   Dropoff Target  : NGO_002
   ETA to Pickup   : ~10 mins
   Routing Logic   : Volunteer VOL_102 is chosen because they are c

# **The Predictive Forecast Module**

Generate Data & Train the Predictive Model

In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import RandomForestRegressor
from datetime import datetime, timedelta

print(">> Initializing Predictive Forecast Engine...")

# 1. Generate Realistic Synthetic Data (Historical Logs)
# Simulating 6 months of data across local hubs
neighborhoods = ["Mangolpuri", "Rohini", "Pitampura", "Saraswati Vihar", "Paschim Vihar"]
np.random.seed(42)

data = []
for _ in range(2000): # 2000 historical surplus events
    neighborhood = np.random.choice(neighborhoods)
    hour = np.random.randint(10, 24) # 10 AM to Midnight
    day_of_week = np.random.randint(0, 7)
    is_weekend = 1 if day_of_week >= 5 else 0
    is_raining = np.random.choice([0, 1], p=[0.8, 0.2]) # 20% chance of rain

    # Logic: Weekends and rain cause more surplus food
    base_surplus = np.random.randint(10, 50)
    if is_weekend: base_surplus += np.random.randint(20, 60)
    if is_raining: base_surplus += np.random.randint(10, 40)

    data.append([neighborhood, hour, day_of_week, is_weekend, is_raining, base_surplus])

df = pd.DataFrame(data, columns=['Neighborhood', 'Hour', 'DayOfWeek', 'IsWeekend', 'IsRaining', 'SurplusServings'])

# 2. Prepare Data for ML
# Convert text neighborhoods into numbers the AI can understand (One-Hot Encoding)
df_encoded = pd.get_dummies(df, columns=['Neighborhood'])

X = df_encoded.drop('SurplusServings', axis=1)
y = df_encoded['SurplusServings']

# 3. Train the Model
print(">> Training Random Forest Regressor on historical data...")
model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X, y)
print("✅ Model Trained Successfully.")

>> Initializing Predictive Forecast Engine...
>> Training Random Forest Regressor on historical data...
✅ Model Trained Successfully.


The React API Handoff (JSON Generator)

In [ ]:
# 1. Define current conditions for the forecast
current_hour = 20 # E.g., 8:00 PM
is_weekend_now = 1
is_raining_now = 0

print(f"\n>> Generating 4-Hour Forecast for React Frontend (Base Hour: {current_hour}:00)...\n")

forecast_results = []

# 2. Predict for each neighborhood
for nhood in neighborhoods:
    # Create an empty row matching our training data features
    input_data = pd.DataFrame(columns=X.columns)
    input_data.loc[0] = 0 # Fill with zeros

    # Set the current conditions
    input_data['Hour'] = current_hour
    input_data['DayOfWeek'] = 5 # Assuming Saturday
    input_data['IsWeekend'] = is_weekend_now
    input_data['IsRaining'] = is_raining_now

    # Activate the specific neighborhood column
    nhood_col = f'Neighborhood_{nhood}'
    if nhood_col in input_data.columns:
        input_data[nhood_col] = 1

    # 3. Generate the prediction
    predicted_surplus = model.predict(input_data)[0]

    # Calculate a simple "Confidence / Hotspot Severity" score (0 to 100%)
    severity_score = min(round((predicted_surplus / 120) * 100), 100)

    # Append to our React API payload
    forecast_results.append({
        "id": nhood.lower().replace(" ", "_"),
        "neighborhood": nhood,
        "predicted_servings": int(predicted_surplus),
        "hotspot_severity_percent": severity_score,
        "recommended_action": "Pre-stage volunteers" if severity_score > 60 else "Monitor"
    })

# 4. Output the exact JSON your React app will consume
react_api_payload = {
    "status": "success",
    "timestamp": str(datetime.now()),
    "forecast": sorted(forecast_results, key=lambda x: x['predicted_servings'], reverse=True)
}

print(json.dumps(react_api_payload, indent=2))


>> Generating 4-Hour Forecast for React Frontend (Base Hour: 20:00)...

{
  "status": "success",
  "timestamp": "2026-07-19 04:47:21.804932",
  "forecast": [
    {
      "id": "saraswati_vihar",
      "neighborhood": "Saraswati Vihar",
      "predicted_servings": 71,
      "hotspot_severity_percent": 59,
      "recommended_action": "Monitor"
    },
    {
      "id": "rohini",
      "neighborhood": "Rohini",
      "predicted_servings": 70,
      "hotspot_severity_percent": 58,
      "recommended_action": "Monitor"
    },
    {
      "id": "pitampura",
      "neighborhood": "Pitampura",
      "predicted_servings": 66,
      "hotspot_severity_percent": 56,
      "recommended_action": "Monitor"
    },
    {
      "id": "paschim_vihar",
      "neighborhood": "Paschim Vihar",
      "predicted_servings": 65,
      "hotspot_severity_percent": 55,
      "recommended_action": "Monitor"
    },
    {
      "id": "mangolpuri",
      "neighborhood": "Mangolpuri",
      "predicted_servings": 56,
   

# **DATABASE CREATION**

Initialize the SQLite Database

In [ ]:
import sqlite3
import pandas as pd

# 1. Create a connection to a local database file in Colab
# This will create a file named 'food_rescue.db' in your Colab files tab
conn = sqlite3.connect('food_rescue.db')
cursor = conn.cursor()

print(">> Initializing Database Schemas...")

# 2. Create the NGOs Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS ngos (
    ngo_id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    capacity_servings INTEGER NOT NULL,
    accepts_cooked BOOLEAN DEFAULT 1,
    location_lat REAL NOT NULL,
    location_lon REAL NOT NULL,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
)
''')

# 3. Create the Volunteers Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS volunteers (
    vol_id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    vehicle_type TEXT NOT NULL,
    status TEXT DEFAULT 'idle',
    current_lat REAL,
    current_lon REAL,
    last_ping DATETIME DEFAULT CURRENT_TIMESTAMP
)
''')

# 4. Create the Surplus Alerts Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS surplus_alerts (
    alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
    food_type TEXT NOT NULL,
    quantity_servings INTEGER NOT NULL,
    time_to_spoil_hours INTEGER NOT NULL,
    location_lat REAL NOT NULL,
    location_lon REAL NOT NULL,
    status TEXT DEFAULT 'pending_match',
    raw_message_text TEXT,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
)
''')

# 5. Create the Dispatch Ledger
cursor.execute('''
CREATE TABLE IF NOT EXISTS dispatch_ledger (
    dispatch_id INTEGER PRIMARY KEY AUTOINCREMENT,
    alert_id INTEGER NOT NULL,
    ngo_id TEXT NOT NULL,
    vol_id TEXT NOT NULL,
    dispatch_urgency TEXT NOT NULL,
    estimated_pickup_mins INTEGER,
    routing_reasoning TEXT,
    delivery_status TEXT DEFAULT 'assigned',
    completed_at DATETIME,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (alert_id) REFERENCES surplus_alerts(alert_id),
    FOREIGN KEY (ngo_id) REFERENCES ngos(ngo_id),
    FOREIGN KEY (vol_id) REFERENCES volunteers(vol_id)
)
''')

# Save changes
conn.commit()
print("✅ Database and Tables Created Successfully.")

>> Initializing Database Schemas...
✅ Database and Tables Created Successfully.


Populate with Mock Data & Query It

In [ ]:
# 1. Insert our active NGOs into the SQL Database
ngo_data = [
    ("NGO_001", "Hope Foundation", 20, 1, 28.7050, 77.1000),
    ("NGO_002", "Delhi Hunger Relief", 200, 1, 28.7100, 77.1150)
]

# Use IGNORE to prevent errors if you run this cell multiple times
cursor.executemany('''
INSERT OR IGNORE INTO ngos (ngo_id, name, capacity_servings, accepts_cooked, location_lat, location_lon)
VALUES (?, ?, ?, ?, ?, ?)
''', ngo_data)

conn.commit()

# 2. Query the database to prove it works
print(">> Fetching active NGOs directly from SQL Database:\n")

# We use Pandas here just to make the SQL output look pretty in Colab
df_ngos = pd.read_sql_query("SELECT * FROM ngos", conn)
print(df_ngos)

>> Fetching active NGOs directly from SQL Database:

    ngo_id                 name  capacity_servings  accepts_cooked  \
0  NGO_001      Hope Foundation                 20               1   
1  NGO_002  Delhi Hunger Relief                200               1   

   location_lat  location_lon           created_at  
0        28.705        77.100  2026-07-19 04:45:28  
1        28.710        77.115  2026-07-19 04:45:28  


# **Wiring the AI Pipeline to the Database**

Populate the Volunteer Fleet in SQL

In [ ]:
import pandas as pd

# 1. Insert our fleet into the SQLite database
volunteer_data = [
    ("VOL_089", "Aman", "bike", "idle", 28.7042, 77.1026),
    ("VOL_102", "Priya", "car", "idle", 28.7060, 77.1040)
]

cursor.executemany('''
INSERT OR IGNORE INTO volunteers (vol_id, name, vehicle_type, status, current_lat, current_lon)
VALUES (?, ?, ?, ?, ?, ?)
''', volunteer_data)

conn.commit()

# 2. Verify the data is in the database
print(">> Fetching idle volunteers directly from SQL:\n")
df_vols = pd.read_sql_query("SELECT * FROM volunteers WHERE status = 'idle'", conn)
print(df_vols)

>> Fetching idle volunteers directly from SQL:

    vol_id  name vehicle_type status  current_lat  current_lon  \
0  VOL_089  Aman         bike   idle      28.7042      77.1026   

             last_ping  
0  2026-07-19 04:45:28  


The Database-Connected Pipeline

In [ ]:
def run_database_driven_pipeline(whatsapp_text: str, current_gps: dict):
    print("="*70)
    print("🚨 NEW SURPLUS ALERT (DATABASE LINKED)")
    print("="*70)

    # --- 1. NLP Extraction & Database Insertion ---
    surplus_data = parse_whatsapp_surplus(whatsapp_text, current_gps)

    # Save the alert to the database
    cursor.execute('''
        INSERT INTO surplus_alerts (food_type, quantity_servings, time_to_spoil_hours, location_lat, location_lon, raw_message_text)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (surplus_data['food_type'], surplus_data['quantity_servings'], surplus_data['time_to_spoil_hours'],
          surplus_data['location_lat'], surplus_data['location_lon'], whatsapp_text))
    conn.commit()
    alert_id = cursor.lastrowid # Get the ID of the row we just inserted
    print(f">> [DB] Alert saved to surplus_alerts table (ID: {alert_id})")

    # --- 2. Query Database for Groq Agents ---
    # Convert SQL tables into the dictionary format the AI expects
    db_ngos = pd.read_sql_query("SELECT * FROM ngos", conn).to_dict('records')
    db_vols = pd.read_sql_query("SELECT * FROM volunteers WHERE status = 'idle'", conn).to_dict('records')

    # --- 3. Run Matching Agent ---
    print(">> Pinging Groq Matching Agent...")
    match_res = json.loads(run_matching_agent_groq(surplus_data, db_ngos))

    if not match_res.get("match_found"):
        print("❌ Terminated: No matching NGO found.")
        return

    best_ngo_id = match_res.get("best_ngo_id")
    urgency = match_res.get("dispatch_urgency")

    # --- 4. Run Dispatch Agent ---
    print(">> Pinging Groq Dispatch Agent...")
    dispatch_res = json.loads(run_dispatch_agent_groq(surplus_data, best_ngo_id, urgency, db_vols))
    volunteer_id = dispatch_res.get('assigned_volunteer_id')

    # --- 5. Finalize Transaction in Database ---
    if volunteer_id:
        # Create the dispatch ledger receipt
        cursor.execute('''
            INSERT INTO dispatch_ledger (alert_id, ngo_id, vol_id, dispatch_urgency, estimated_pickup_mins, routing_reasoning)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (alert_id, best_ngo_id, volunteer_id, urgency, dispatch_res.get('estimated_pickup_mins'), dispatch_res.get('reasoning')))

        # Mark the volunteer as busy so they don't get double-booked
        cursor.execute("UPDATE volunteers SET status = 'en_route_pickup' WHERE vol_id = ?", (volunteer_id,))

        # Update the alert status
        cursor.execute("UPDATE surplus_alerts SET status = 'matched' WHERE alert_id = ?", (alert_id,))
        conn.commit()

        print("\n✅ SUCCESS: LEDGER UPDATED & DRIVER DISPATCHED")
        print(f"   Driver Assigned: {volunteer_id}")
        print(f"   Dropoff Target : {best_ngo_id}")
    else:
        print("\n⚠️ ALERT: Match found, but no suitable volunteers are active.")
    print("="*70 + "\n")

# --- Execute the Live Test ---
test_text = "urgent: 150 boxes of paneer tikka remaining from the wedding party. pick up now."
test_gps = {"lat": 28.7120, "lon": 77.1200}

run_database_driven_pipeline(test_text, test_gps)

🚨 NEW SURPLUS ALERT (DATABASE LINKED)
>> [DB] Alert saved to surplus_alerts table (ID: 2)
>> Pinging Groq Matching Agent...
>> Pinging Groq Dispatch Agent...

✅ SUCCESS: LEDGER UPDATED & DRIVER DISPATCHED
   Driver Assigned: VOL_089
   Dropoff Target : NGO_002



Tranfer all data into EXCEL SHEET

In [ ]:
import pandas as pd
from google.colab import files

# Install xlsxwriter if it's not already installed
try:
    import xlsxwriter
except ImportError:
    %pip install xlsxwriter
    import xlsxwriter

print(">> Extracting SQL Database to Excel...")

# 1. Query all tables into separate Pandas DataFrames
df_ngos = pd.read_sql_query("SELECT * FROM ngos", conn)
df_volunteers = pd.read_sql_query("SELECT * FROM volunteers", conn)
df_alerts = pd.read_sql_query("SELECT * FROM surplus_alerts", conn)
df_ledger = pd.read_sql_query("SELECT * FROM dispatch_ledger", conn)

# 2. Create the Excel file and write each DataFrame to a separate sheet
excel_filename = "food_rescue_database_export.xlsx"

with pd.ExcelWriter(excel_filename, engine='xlsxwriter') as writer:
    df_alerts.to_excel(writer, sheet_name='Surplus Alerts', index=False)
    df_ledger.to_excel(writer, sheet_name='Dispatch Ledger', index=False)
    df_ngos.to_excel(writer, sheet_name='NGOs', index=False)
    df_volunteers.to_excel(writer, sheet_name='Volunteers', index=False)

print(f"✅ Data successfully written to {excel_filename}")

# 3. Trigger Google Colab to download the file to your local machine
files.download(excel_filename)

>> Extracting SQL Database to Excel...
✅ Data successfully written to food_rescue_database_export.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Flask API**

The Flask Server & API Logic

In [ ]:
!pip install flask flask-cors

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading
import json
import pandas as pd

app = Flask(__name__)
CORS(app) # Allows your React frontend to make requests to this API

def process_surplus_for_api(whatsapp_text, current_gps):
    """Modified pipeline that returns a dictionary for the API response."""
    # 1. NLP Extraction & DB Insert
    surplus_data = parse_whatsapp_surplus(whatsapp_text, current_gps)

    cursor.execute('''
        INSERT INTO surplus_alerts (food_type, quantity_servings, time_to_spoil_hours, location_lat, location_lon, raw_message_text)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (surplus_data['food_type'], surplus_data['quantity_servings'], surplus_data['time_to_spoil_hours'],
          surplus_data['location_lat'], surplus_data['location_lon'], whatsapp_text))
    conn.commit()
    alert_id = cursor.lastrowid

    # 2. Match NGO
    db_ngos = pd.read_sql_query("SELECT * FROM ngos", conn).to_dict('records')
    match_res = json.loads(run_matching_agent_groq(surplus_data, db_ngos))

    if not match_res.get("match_found"):
        return {"status": "failed", "message": "No matching NGO found."}

    best_ngo_id = match_res.get("best_ngo_id")
    urgency = match_res.get("dispatch_urgency")

    # 3. Dispatch Volunteer
    db_vols = pd.read_sql_query("SELECT * FROM volunteers WHERE status = 'idle'", conn).to_dict('records')
    dispatch_res = json.loads(run_dispatch_agent_groq(surplus_data, best_ngo_id, urgency, db_vols))
    volunteer_id = dispatch_res.get('assigned_volunteer_id')

    if volunteer_id:
        # Finalize Ledger
        cursor.execute('''
            INSERT INTO dispatch_ledger (alert_id, ngo_id, vol_id, dispatch_urgency, estimated_pickup_mins, routing_reasoning)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (alert_id, best_ngo_id, volunteer_id, urgency, dispatch_res.get('estimated_pickup_mins'), dispatch_res.get('reasoning')))

        cursor.execute("UPDATE volunteers SET status = 'en_route_pickup' WHERE vol_id = ?", (volunteer_id,))
        cursor.execute("UPDATE surplus_alerts SET status = 'matched' WHERE alert_id = ?", (alert_id,))
        conn.commit()

        # Build successful JSON response
        return {
            "status": "success",
            "data": {
                "food_type": surplus_data['food_type'],
                "quantity": surplus_data['quantity_servings'],
                "assigned_volunteer": volunteer_id,
                "vehicle": dispatch_res.get('vehicle_type'),
                "target_ngo": best_ngo_id,
                "eta_mins": dispatch_res.get('estimated_pickup_mins'),
                "logic": dispatch_res.get('reasoning')
            }
        }
    else:
        return {"status": "failed", "message": "Match found, but no active volunteers available."}

# Define the POST endpoint
@app.route('/api/dispatch', methods=['POST'])
def dispatch_endpoint():
    try:
        req_data = request.get_json()
        text = req_data.get('text', '')
        gps = req_data.get('gps', {"lat": 28.7041, "lon": 77.1025})

        result = process_surplus_for_api(text, gps)
        return jsonify(result)

    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 500

# Start Flask in a background thread
def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_flask).start()
print("✅ Flask API is now running in the background on port 5000.")

✅ Flask API is now running in the background on port 5000.


Expose the API to the Internet

In [ ]:
# Get the public IP of your Colab instance (needed for localtunnel security)
!curl -s ipv4.icanhazip.com

# Start the tunnel
!npx localtunnel --port 5000

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


8.229.139.28
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙your url is: https://itchy-beers-leave.loca.lt
^C
